# ResNet学習用 crop画像作成パイプライン（ローカル/VSCode版・バッファ付き）

`colab/crystal_dataset_pipeline.ipynb`のローカル版。Google Driveのマウントは不要で、
元データ（画像・CSV）がローカルにあれば、このnotebookだけで`vscode_project/data_buffered/`まで直接作成できる。

**今までの`data/`フォルダは一切上書きしない。** 出力先は新しく`data_buffered/`にしてあるので、
既存の学習済みモデル・データセットはそのまま安全に残る。

既存版との違いは、矩形をそのまま切り出すのではなく、**`BUFFER_PX`分だけ矩形を広げてから
cropする**こと（YOLOの検出boxが多少緩くても、ResNetが頑健に読み取れるようにする狙い）。

`vscode_project/`フォルダをカレントディレクトリとしてこのnotebookを実行すること。

## Step 0: セットアップ

In [ ]:
import os
import re
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image

# ===== パス設定（実際のフォルダパスに変更） =====
IMAGE_ROOT = r"path/to/画像データ"
CSV_ROOT   = r"path/to/Excelデータ"

# ===== 出力先（新規。既存のdata/は上書きしない） =====
OUTPUT_DIR   = "data_buffered/crops"
DATASET_CSV  = "data_buffered/dataset.csv"
CLEAN_DATASET_CSV = "data_buffered/dataset_clean.csv"

# ===== バッファ（この実験の肝。pxは40倍相当で指定） =====
BUFFER_PX = 10  # 矩形の上下左右にこれだけ広げてからcropする（0にすれば従来と同じ）

COL_X1 = "矩形領域(左上:x)"
COL_Y1 = "矩形領域(左上:y)"
COL_X2 = "矩形領域(右下:x)"
COL_Y2 = "矩形領域(右下:y)"
COL_DIAMETER = "円相当径"
IMG_EXTS = (".bmp", ".tif", ".tiff", ".jpg", ".png")
EXCLUDE_MINUTES = {"0"}
MAG_TO_UM_PER_PIXEL = {40: 0.088725, 20: 0.17353, 10: 0.34392}
BASE_UM_PER_PIXEL = MAG_TO_UM_PER_PIXEL[40]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"出力先: {OUTPUT_DIR}")
print(f"BUFFER_PX = {BUFFER_PX} (40倍相当px)")

## Step 1: 画像・CSVの対応付け ＋ crop作成（バッファ付き）

In [ ]:
def normalize_folder_name(name):
    return re.sub(r"[（）()・\s]", "", name)

def get_minute_mag_key(filename):
    m = re.search(r"(\d+)分.*?(\d+)倍", filename)
    return m.groups() if m else None

def find_diameter_col(columns):
    for c in columns:
        if COL_DIAMETER in str(c):
            return c
    raise ValueError(f"円相当径列が見つかりません: {list(columns)}")


def make_dataset():
    img_dirs = {normalize_folder_name(d): d
                for d in os.listdir(IMAGE_ROOT)
                if os.path.isdir(os.path.join(IMAGE_ROOT, d))}
    csv_dirs = {normalize_folder_name(d): d
                for d in os.listdir(CSV_ROOT)
                if os.path.isdir(os.path.join(CSV_ROOT, d))}

    common = set(img_dirs) & set(csv_dirs)
    print(f"画像フォルダ: {len(img_dirs)}件  CSVフォルダ: {len(csv_dirs)}件  対応: {len(common)}件")

    rows, crop_id = [], 0

    for key in sorted(common):
        image_dir = os.path.join(IMAGE_ROOT, img_dirs[key])
        csv_dir   = os.path.join(CSV_ROOT,   csv_dirs[key])
        print(f"\n=== {img_dirs[key]} ===")

        images = defaultdict(list)
        for f in os.listdir(image_dir):
            if f.lower().endswith(IMG_EXTS):
                k = get_minute_mag_key(f)
                if k:
                    images[k].append(f)

        csvs = defaultdict(list)
        for f in os.listdir(csv_dir):
            if f.lower().endswith(".csv"):
                k = get_minute_mag_key(f)
                if k:
                    csvs[k].append(f)

        for k in sorted(set(images) & set(csvs)):
            minute, _ = k
            if minute in EXCLUDE_MINUTES:
                print(f"  [除外] {minute}分")
                continue

            for img_f, csv_f in zip(sorted(images[k]), sorted(csvs[k])):
                image_path = os.path.join(image_dir, img_f)
                csv_path   = os.path.join(csv_dir,   csv_f)

                m = re.search(r"(\d+)倍", img_f)
                mag = int(m.group(1)) if m else None
                if mag not in MAG_TO_UM_PER_PIXEL:
                    continue
                um_per_pixel = MAG_TO_UM_PER_PIXEL[mag]
                # バッファはum_per_pixelの座標系(切り出し前の生px)で加える
                buffer_raw_px = BUFFER_PX * (BASE_UM_PER_PIXEL / um_per_pixel)

                try:
                    df  = pd.read_csv(csv_path, encoding="cp932")
                    dcol = find_diameter_col(df.columns)
                    img  = Image.open(image_path)
                except Exception as e:
                    print(f"  [スキップ] {e}")
                    continue
                W, H = img.size

                n_crops = 0
                for _, crystal in df.iterrows():
                    try:
                        x1 = int(crystal[COL_X1]); y1 = int(crystal[COL_Y1])
                        x2 = int(crystal[COL_X2]); y2 = int(crystal[COL_Y2])
                        d  = float(crystal[dcol])
                    except (KeyError, ValueError, TypeError):
                        continue

                    left, right = sorted([x1, x2])
                    top, bottom = sorted([y1, y2])
                    if right <= left or bottom <= top:
                        continue

                    # ここでバッファを加える（画像の端は越えないようclip）
                    left   = max(0, left - buffer_raw_px)
                    top    = max(0, top - buffer_raw_px)
                    right  = min(W, right + buffer_raw_px)
                    bottom = min(H, bottom + buffer_raw_px)

                    crop  = img.crop((left, top, right, bottom))
                    scale = um_per_pixel / BASE_UM_PER_PIXEL
                    crop  = crop.resize((max(1, round(crop.width  * scale)),
                                         max(1, round(crop.height * scale))))

                    crop_path = os.path.join(OUTPUT_DIR, f"crop_{crop_id:06d}.png")
                    crop.save(crop_path)

                    rows.append({"filepath": os.path.abspath(crop_path),  # 絶対パス。load_valid_datasetがdata/cropsを見に行かないようにするため
                                 "label_um": d * um_per_pixel,
                                 "source_folder": img_dirs[key],
                                 "source_image":  img_f})
                    crop_id += 1
                    n_crops += 1

                print(f"  {img_f} → {n_crops}件")

    out_df = pd.DataFrame(rows)
    out_df.to_csv(DATASET_CSV, index=False, encoding="utf-8-sig")
    print(f"\n完了: {len(out_df)}件 → {DATASET_CSV}")

make_dataset()

## Step 2: フィルタ（形状・カラーアーティファクト除去）

In [ ]:
# ===== 形状フィルタ =====
MIN_SIZE   = 10   # 幅・高さの最小ピクセル数
MIN_AREA   = 200  # 面積(px^2)の最小値
MAX_ASPECT = 10   # 長辺/短辺 の最大値

def is_valid_shape(filepath):
    try:
        img = Image.open(filepath)
        w, h = img.size
        if w < MIN_SIZE or h < MIN_SIZE:
            return False
        if w * h < MIN_AREA:
            return False
        if max(w, h) / max(min(w, h), 1) > MAX_ASPECT:
            return False
        return True
    except Exception:
        return False

df = pd.read_csv(DATASET_CSV)
print(f"フィルタ前: {len(df)}件")

valid = df["filepath"].apply(lambda fp: is_valid_shape(os.path.join(OUTPUT_DIR, fp)))
df_filtered = df[valid].reset_index(drop=True)
print(f"形状フィルタ後: {len(df_filtered)}件 (除外: {(~valid).sum()}件)")

# ===== カラーアーティファクト除去 =====
CHANNEL_DIFF_THRESHOLD = 14

def channel_diff(filepath):
    try:
        img = np.array(Image.open(filepath))
        if img.ndim == 2:
            return 0.0
        return float(np.std(img.mean(axis=(0, 1))))
    except Exception:
        return 0.0

df_filtered["channel_diff"] = df_filtered["filepath"].apply(
    lambda fp: channel_diff(os.path.join(OUTPUT_DIR, fp))
)

df_clean = df_filtered[df_filtered["channel_diff"] <= CHANNEL_DIFF_THRESHOLD].reset_index(drop=True)
removed = len(df_filtered) - len(df_clean)
print(f"カラーアーティファクト除去後: {len(df_clean)}件 (除外: {removed}件)")

df_clean.to_csv(CLEAN_DATASET_CSV, index=False, encoding="utf-8-sig")
print(f"保存: {CLEAN_DATASET_CSV}")